I used Logistic Regression, Random Forests, Lightgbm and Support Vector Machine as models. I get the highest score is 85.30% with lightgbm.

In [ ]:
import numpy as np
import pandas as pd

train_path = "/kaggle/input/titanic/train.csv"
test_path = "/kaggle/input/titanic/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Data Cleaning And Feature Engineering

In [ ]:
# make copies of train and test sets
train_copy = train_df.copy()
test_copy = test_df.copy()

In [ ]:
# preparing sets
train_copy = train_copy.drop("Survived", axis = 1)
train_copy = train_copy.drop("PassengerId", axis = 1)


test_copy = test_copy.drop("PassengerId", axis = 1)

In [ ]:
#fill "Age" column with median
median_age = train_df["Age"].median()

train_copy["Age"] = train_copy["Age"].fillna(median_age)
test_copy["Age"] = test_copy["Age"].fillna(median_age)


In [ ]:
# there is a missing fare value in test set
fare_median = train_copy["Fare"].median()

test_copy["Fare"] = test_copy["Fare"].fillna(fare_median)

In [ ]:
### Why drop 'Ticket'? The 'Ticket' column is very messy, with mixed numbers and letters (like 'A/5 21171' or '347082'). Extracting useful information from it is very complex (advanced feature engineering). For this model, the complexity isn't worth the small potential gain, so we are dropping it.
train_copy = train_copy.drop("Ticket", axis = 1)
test_copy = test_copy.drop("Ticket", axis = 1)

In [ ]:
# I filled "Embarked " column with "S" because there were few null values in the column
train_copy["Embarked"] = train_copy["Embarked"].fillna("S")
test_copy["Embarked"] = test_copy["Embarked"].fillna("S")

In [ ]:
# I used get_dummies method for "Sex"and "Embarked"
cat_nums = ["Sex", "Embarked"]

train_copy = pd.get_dummies(train_copy, columns = cat_nums, dtype = int)
test_copy = pd.get_dummies(test_copy, columns = cat_nums, dtype = int)


train_copy.info()

In [ ]:
#We take the space, letter and dot in front of the name for new feature
for dataset in (train_copy, test_copy):
    dataset["Title"] = dataset["Name"].str.extract(" ([A-Za-z]+)\.")

print(train_copy['Title'].value_counts())

In [ ]:
#Because the title values are so diverse, we group them
for dataset in (train_copy, test_copy):
    dataset["Title"] = dataset["Title"].replace(['Mlle', 'Ms'], 'Miss')
    dataset["Title"] = dataset["Title"].replace('Mme', 'Mrs')
    dataset['Title'] = dataset['Title'].replace(
        ['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 
        'Rare'
    )

print(train_copy['Title'].value_counts())

In [ ]:
# now we do not need "Name" column and "Title" column ready for one hot encoding
train_copy = train_copy.drop("Name", axis = 1)
test_copy = test_copy.drop("Name", axis = 1)

title_column = ["Title"]

train_copy = pd.get_dummies(train_copy, columns = title_column , dtype = int)
test_copy = pd.get_dummies(test_copy, columns = title_column, dtype = int)

In [ ]:
# we create 2 new features. If person not alone "IsAlone" is equal to 1 otherwise 0
train_copy['FamilySize'] = train_copy['SibSp'] + train_copy['Parch'] + 1
test_copy['FamilySize'] = test_copy['SibSp'] + test_copy['Parch'] + 1

train_copy['IsAlone'] = 0
train_copy.loc[train_copy['FamilySize'] == 1, "IsAlone"] = 1

test_copy['IsAlone'] = 0
test_copy.loc[test_copy['FamilySize'] == 1, "IsAlone"] = 1

In [ ]:
# we create a new feature "Deck" because The first letter of the deck gives information about where the person is staying
train_copy["Deck"] = train_copy["Cabin"].fillna("U").str[0]
test_copy["Deck"] = test_copy["Cabin"].fillna("U").str[0]

train_copy = train_copy.drop("Cabin", axis = 1)
test_copy = test_copy.drop("Cabin", axis = 1)


deck_column = ["Deck"]

train_copy = pd.get_dummies(train_copy, columns = deck_column, dtype = int)
test_copy = pd.get_dummies(test_copy, columns = deck_column, dtype = int)

In [ ]:
#preparing sets for training a model
x_train = train_copy
y_train = train_df["Survived"]

x_test = test_copy

In [ ]:
### Critical Step: Aligning Train and Test Columns When we used pd.get_dummies() (e.g., for the 'Deck' feature), the train set might have had a 'Deck_T' which the test set didn't. This creates a mismatch in the number of columns. To fix this, we use .reindex() to force the x_test columns to match x_train's columns exactly, filling any missing new columns (like 'Deck_T') with 0.
train_columns = x_train.columns

x_test = x_test.reindex(columns=train_columns, fill_value=0)

# Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

#Cross Validation
X_train, X_val, Y_train, Y_val = train_test_split(
    x_train,
    y_train,
    test_size = 0.2,
    random_state = 42
)

#Logistic Regression model
log_model = LogisticRegression(max_iter =  10000)

log_model.fit(X_train, Y_train)

log_predictions = log_model.predict(X_val)

scor = accuracy_score(Y_val,log_predictions)

print(f"Validation Score: {scor * 100:.2f}%")

In [ ]:
from sklearn.model_selection import cross_val_score

# Cross Validation model
scores = cross_val_score(log_model, x_train, y_train, cv=5, scoring='accuracy')

print(f"CV scores: {scores}")
print(f"Mean of scores: {scores.mean() * 100:.2f}%")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#Random forest model
forest_reg = RandomForestClassifier(n_estimators = 100, max_depth = 5, random_state = 42)

forest_reg.fit(X_train, Y_train)

forest_predicts = forest_reg.predict(X_val)

forest_score = accuracy_score(Y_val, forest_predicts)

print(f"RF score: {forest_score * 100:.2f}%")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

#we make a dict for randomizedsearch
params = {
    "n_estimators" : [100, 200, 300, 400, 500, 600],
    "max_depth" : [5, 10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],                
    'min_samples_leaf': [1, 2, 4],                  
    'criterion': ['gini', 'entropy']
}

#random forest model
rf_model = RandomForestClassifier(random_state=42)

#randomizedsearch model
random_search = RandomizedSearchCV(
    estimator = rf_model,
    n_iter = 100,
    param_distributions=params,
    cv=5, 
    random_state=42, 
    n_jobs=-1,
    scoring='accuracy'
)

random_search.fit(x_train, y_train)

print(f"The best score: {random_search.best_score_ * 100:.2f}%")
print("Best parameters:")
print(random_search.best_params_)

best_model = random_search.best_estimator_

In [ ]:
from lightgbm import LGBMClassifier

#Lightgbm model
lght_model = LGBMClassifier(random_state = 42, verbose = -1)

lght_model.fit(X_train, Y_train)
lght_predicts = lght_model.predict(X_val)
lght_scores = accuracy_score(Y_val, lght_predicts)

print(f"Lightgbm score : {lght_scores * 100:.2f}%")

In [ ]:
#we make a dict for randomizedsearch
param_dist = {
    
    'n_estimators': [100, 200, 300, 400, 500, 700, 1000],
    
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.1, 0.15],

    'num_leaves': [10, 15, 20, 25, 31, 40],
    
    'max_depth': [5, 7, 10, 15, -1],

    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    
    'reg_alpha': [0, 0.1, 0.5, 1],
    'reg_lambda': [0, 0.1, 0.5, 1]
}

#Lightgbm model
lght_model = LGBMClassifier(force_col_wise=True, random_state = 42, verbose = -1)

#RandomizedSearch model
random_search2 = RandomizedSearchCV(
    estimator = lght_model,
    n_iter = 100,
    param_distributions=param_dist,
    cv=5, 
    random_state=42, 
    n_jobs=4,
    scoring='accuracy',
    verbose = 1
)

random_search2.fit(x_train, y_train)

print(f"Best score of Lightgbm: {random_search2.best_score_ * 100:.2f}%")
print("Best parameters")
print(random_search2.best_params_)

lght_best_model = random_search2.best_estimator_

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

### Why use StandardScaler for SVC? Models like RandomForest and LightGBM (tree-based) don't care if one feature is on a scale of 0-1 (like 'Sex_male') and another is 0-500 (like 'Fare'). However, models like SVM (Support Vector Machine) and KNN are distance-based. They ARE very sensitive to different scales. We use StandardScaler to put all features on the same scale (mean=0, std=1) so that 'Fare' doesn't unfairly dominate the model's logic.
scaler = StandardScaler()

scaled_train = scaler.fit_transform(X_train)

scaled_val = scaler.transform(X_val)

scaled_test = scaler.transform(x_test)

svm_model = SVC(random_state = 42)

svm_model.fit(scaled_train, Y_train)

svm_predicts = svm_model.predict(scaled_val)

svm_scores = accuracy_score(Y_val, svm_predicts)

print(f"SVM score : {svm_scores * 100:.2f}%")